# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate all record sets in the dataset by their `@id`, and list the fields for each record set. By referencing everything by `@id`, we ensure consistent referencing throughout the notebook.

In [ ]:
# List all available record sets and their fields.

print("Available record sets in the dataset:")
for rs in dataset.record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - Field @id: {field.get('@id')}, name: {field.get('name')}")
        else:  # field might actually just be a string @id
            print(f"    - Field @id: {field}")
print("\nExample of iterating through records of the first record set:\n")
if dataset.record_sets:
    first_rs_id = dataset.record_sets[0]['@id']
    for idx, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if idx >= 2:
            print("...")
            break

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s discovered in the previous step.

In [ ]:
# Extract data from each record set, referenced by @id

record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows from RecordSet @id: {record_set_id}")

# Show columns of the first record set
if record_sets:
    first_rs_id = record_sets[0]
    print(f"\nColumns of RecordSet @id '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    print("\nSample data:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Here, we demonstrate these steps for one numeric field, referencing all entities via their `@id`.

In [ ]:
# Example: Filter for records where the 'age_at_second_crc_diagnosis' (referenced by its @id) > 60
# and normalize the values. Replace with actual @id from the data overview as needed.

# Identify a numeric field by its @id in your record set. For demonstration,
# let's assume the field with @id 'https://api.app.sen.science/frontiers/7862866/age2' represents age at second CRC diagnosis.
# Adjust the @id and record set id to your data if different.

record_set_id = record_sets[0]  # Use the first record set as example
numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field by looking for 'age' or 'years' in columns
df = dataframes[record_set_id]
for col in df.columns:
    if 'age' in col.lower() or 'year' in col.lower():
        numeric_field_id = col
        break
# Similarly, try to find a suitable group field, e.g. 'sex' or 'gender' or 'msi' in columns
for col in df.columns:
    if 'sex' in col.lower() or 'gender' in col.lower() or 'msi' in col.lower():
        group_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field identified. Please update numeric_field_id to a @id from your dataset.")
else:
    print(f"Processing field: {numeric_field_id} (@id)")
    # Drop NA and convert to float if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 60
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id (e.g. 'sex', 'msi_status', etc.)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the normalized numeric field and other relevant relationships, referencing the field(s) by their `@id`.

In [ ]:
# Simple histogram and boxplot of the selected numeric field, referencing by @id
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric field detected for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library. We:
- Loaded the dataset directly from its Croissant schema URL;
- Explored available record sets, fields, and respective `@id`s;
- Extracted and processed tabular data by referencing fields and record sets via their `@id`;
- Performed exploratory data analysis, such as filtering and normalization;
- Visualized key data distributions and groupings.

The use of `@id` throughout ensures your data wrangling scripts stay robust and consistent. For deeper analysis, consider investigating relationships between molecular (e.g., MSI-H) and clinicopathological variables, and consult the FAIR^2 documentation for more field information.